<a href="https://colab.research.google.com/github/flango2023/Pulso_F2/blob/main/fase2_parte1_extracao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pulso_Fase_2: Extração de Sintomas e Sugestão de Diagnóstico

**Projeto:** Pulso:  Inteligência Cardiológica com Consciência de Dados  
**Autor:** Richard Schmitz | RM 567951  


---

## Objetivo

Este notebook implementa a Parte 1 da Fase 2 do projeto Pulso.  
A partir de relatos textuais de pacientes, o sistema:

1. Lê as frases do arquivo `sintomas_pacientes.txt`
2. Identifica sintomas com base no mapa de conhecimento (`mapa_conhecimento.csv`)
3. Sugere um possível diagnóstico para cada relato
4. Analisa se há diferença na identificação entre sintomas típicos (masculinos) e atípicos (femininos)

O eixo de viés de gênero documentado na Fase 1 orienta a construção dos dados e a análise dos resultados.

## 1. Configuração do ambiente

> **Google Colab:** faça upload de `sintomas_pacientes.txt` e `mapa_conhecimento.csv` antes de executar.

In [1]:
import os
import re
import pandas as pd

# Detecta se está rodando no Google Colab ou localmente
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

# Define caminhos de acordo com o ambiente
if COLAB:
    # No Colab, os arquivos ficam em /content/ após upload
    CAMINHO_FRASES = '/content/sintomas_pacientes.txt'
    CAMINHO_MAPA   = '/content/mapa_conhecimento.csv'
    CAMINHO_SAIDA  = '/content/resultados_diagnostico.csv'
else:
    # Localmente, os arquivos estão em ../data/
    CAMINHO_FRASES = '../data/sintomas_pacientes.txt'
    CAMINHO_MAPA   = '../data/mapa_conhecimento.csv'
    CAMINHO_SAIDA  = '../data/resultados_diagnostico.csv'

print(f'Ambiente: {"Google Colab" if COLAB else "Local"}')
print(f'Frases  : {CAMINHO_FRASES}')
print(f'Mapa    : {CAMINHO_MAPA}')

Ambiente: Google Colab
Frases  : /content/sintomas_pacientes.txt
Mapa    : /content/mapa_conhecimento.csv


## 2. Leitura dos relatos de pacientes

In [8]:
def carregar_frases(caminho):
    """Lê o arquivo de relatos e retorna apenas as linhas numeradas."""
    frases = []
    with open(caminho, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            # Ignora linhas de comentário (começam com #) e linhas vazias
            if linha and not linha.startswith('#'):
                # Remove o número inicial: "1. ", "2. ", etc.
                texto = re.sub(r'^\d+\.\s*', '', linha)
                frases.append(texto)
    return frases

frases = carregar_frases(CAMINHO_FRASES)

print(f'Total de relatos carregados: {len(frases)}\n')
for i, frase in enumerate(frases, 1):
    print(f'Paciente {i:02d}: {frase}')

Total de relatos carregados: 10

Paciente 01: Há dois dias sinto uma dor forte no peito que irradia para o braço esquerdo e piora quando faço qualquer esforço físico.
Paciente 02: Acordo com falta de ar durante a noite e sinto o coração acelerado sem motivo aparente, como se fosse sair do peito.
Paciente 03: Tenho sentido uma pressão intensa no tórax que dura alguns minutos e passa sozinha, mas voltou três vezes essa semana.
Paciente 04: Sinto tontura e suor frio sempre que subo escadas, e ontem quase desmaiei no trabalho.
Paciente 05: Meu coração fica irregular às vezes, como se pulasse uma batida, e sinto um aperto no peito que me deixa com medo.
Paciente 06: Estou com cansaço extremo há duas semanas, mesmo dormindo bem, e não consigo terminar minhas tarefas do dia sem precisar parar para descansar.
Paciente 07: Sinto náusea constante e uma dor estranha nas costas e no maxilar que aparece e some, sem relação com alimentação.
Paciente 08: Tenho ficado com falta de ar ao fazer coisas s

## 3. Carregamento do mapa de conhecimento

In [9]:
# Carrega o CSV com as associações sintoma → doença
mapa = pd.read_csv(CAMINHO_MAPA)

print(f'Mapa de conhecimento: {len(mapa)} associações carregadas\n')
mapa.head(10)

Mapa de conhecimento: 20 associações carregadas



,sintoma_1,sintoma_2,sintoma_3,doenca_associada,nivel_risco,perfil_predominante
0,dor no peito,irradiação para o braço,suor frio,Infarto Agudo do Miocárdio,alto,masculino típico
1,aperto no tórax,pressão no peito,dor que passa sozinha,Angina Instável,alto,masculino típico
2,falta de ar,coração acelerado,palpitações,Arritmia Cardíaca,alto,masculino típico
3,tontura,suor frio,quase desmaio,Síncope Cardíaca,alto,masculino típico
4,coração irregular,batida pulando,aperto no peito,Fibrilação Atrial,alto,masculino típico
5,cansaço extremo,fadiga persistente,sem energia,Insuficiência Cardíaca,alto,feminino atípico
6,náusea,dor no maxilar,dor nas costas,Infarto Agudo do Miocárdio,alto,feminino atípico
7,falta de ar ao esforço leve,tornozelos inchados,inchaço nas pernas,Insuficiência Cardíaca,alto,feminino atípico
8,fadiga,sensação de peso no estômago,mal-estar geral,Doença Arterial Coronariana,alto,feminino atípico
9,desconforto no peito,dificuldade para dormir,pressão leve,Angina Estável,médio,feminino atípico


## 4. Motor de extração de sintomas e sugestão de diagnóstico

A lógica funciona em três etapas:

1. Para cada relato, verifica quais sintomas do mapa estão presentes no texto
2. Conta quantos sintomas de cada doença foram encontrados (pontuação)
3. Sugere a doença com maior pontuação

In [10]:
def normalizar(texto):
    """Converte para minúsculas e remove espaços extras para comparação."""
    return texto.lower().strip()


def extrair_sintomas(frase, mapa):
    """
    Verifica quais sintomas do mapa estão presentes na frase.
    Retorna lista de sintomas únicos encontrados.
    """
    frase_norm = normalizar(frase)
    sintomas_encontrados = []
    colunas_sintoma = ['sintoma_1', 'sintoma_2', 'sintoma_3']

    for _, linha in mapa.iterrows():
        for col in colunas_sintoma:
            sintoma = normalizar(str(linha[col]))
            # Verifica se o sintoma aparece como substring na frase
            if sintoma in frase_norm:
                sintomas_encontrados.append(sintoma)

    return list(set(sintomas_encontrados))  # remove duplicatas


def sugerir_diagnostico(frase, mapa):
    """
    Conta correspondências por doença e retorna a mais provável.
    Retorna: (diagnóstico, nível de risco, perfil predominante, sintomas encontrados)
    """
    frase_norm = normalizar(frase)
    pontuacao = {}  # {doenca: {score, nivel_risco, perfil}}
    colunas_sintoma = ['sintoma_1', 'sintoma_2', 'sintoma_3']

    for _, linha in mapa.iterrows():
        doenca = linha['doenca_associada']
        # Inicializa a doença no dicionário se ainda não existe
        if doenca not in pontuacao:
            pontuacao[doenca] = {
                'score': 0,
                'nivel_risco': linha['nivel_risco'],
                'perfil': linha['perfil_predominante']
            }
        # Incrementa o score para cada sintoma encontrado na frase
        for col in colunas_sintoma:
            sintoma = normalizar(str(linha[col]))
            if sintoma in frase_norm:
                pontuacao[doenca]['score'] += 1

    # Mantém apenas doenças com ao menos uma correspondência
    candidatos = {k: v for k, v in pontuacao.items() if v['score'] > 0}

    if not candidatos:
        return 'Não identificado', 'indefinido', 'indefinido', []

    # Seleciona a doença com maior pontuação
    melhor = max(candidatos, key=lambda k: candidatos[k]['score'])
    sintomas = extrair_sintomas(frase, mapa)

    return (
        melhor,
        candidatos[melhor]['nivel_risco'],
        candidatos[melhor]['perfil'],
        sintomas
    )


print('Motor de diagnóstico carregado.')

Motor de diagnóstico carregado.


## 5. Processamento dos 10 relatos

In [11]:
resultados = []

# Processa cada relato e armazena os resultados
for i, frase in enumerate(frases, 1):
    diagnostico, risco, perfil, sintomas = sugerir_diagnostico(frase, mapa)
    resultados.append({
        'paciente': i,
        'relato': frase,
        'sintomas_identificados': ', '.join(sintomas) if sintomas else 'nenhum',
        'diagnostico_sugerido': diagnostico,
        'nivel_risco': risco,
        'perfil_predominante': perfil
    })

df_resultados = pd.DataFrame(resultados)

# Exibição formatada de cada resultado
for _, row in df_resultados.iterrows():
    print(f"{'='*70}")
    print(f"Paciente {row['paciente']:02d}")
    print(f"Relato    : {row['relato']}")
    print(f"Sintomas  : {row['sintomas_identificados']}")
    print(f"Diagnóstico sugerido : {row['diagnostico_sugerido']}")
    print(f"Nível de risco       : {row['nivel_risco']}")
    print(f"Perfil predominante  : {row['perfil_predominante']}")
print('='*70)

Paciente 01
Relato    : Há dois dias sinto uma dor forte no peito que irradia para o braço esquerdo e piora quando faço qualquer esforço físico.
Sintomas  : nenhum
Diagnóstico sugerido : Não identificado
Nível de risco       : indefinido
Perfil predominante  : indefinido
Paciente 02
Relato    : Acordo com falta de ar durante a noite e sinto o coração acelerado sem motivo aparente, como se fosse sair do peito.
Sintomas  : coração acelerado, falta de ar
Diagnóstico sugerido : Arritmia Cardíaca
Nível de risco       : alto
Perfil predominante  : masculino típico
Paciente 03
Relato    : Tenho sentido uma pressão intensa no tórax que dura alguns minutos e passa sozinha, mas voltou três vezes essa semana.
Sintomas  : nenhum
Diagnóstico sugerido : Não identificado
Nível de risco       : indefinido
Perfil predominante  : indefinido
Paciente 04
Relato    : Sinto tontura e suor frio sempre que subo escadas, e ontem quase desmaiei no trabalho.
Sintomas  : suor frio, tontura
Diagnóstico sugerido : 

## 6. Análise de viés de gênero na extração de sintomas

Esta seção verifica se o sistema identifica sintomas com a mesma eficácia  
para relatos com perfil masculino típico (frases 1–5) e feminino atípico (frases 6–10).  

Essa análise conecta diretamente com o eixo de governança documentado na Fase 1:  
sistemas treinados com dados majoritariamente masculinos tendem a identificar  
sintomas atípicos femininos com menor precisão.

In [12]:
# Classifica cada paciente por grupo de gênero com base na posição no arquivo
df_resultados['grupo'] = df_resultados['paciente'].apply(
    lambda x: 'Masculino típico' if x <= 5 else 'Feminino atípico'
)

# Conta quantos sintomas foram identificados por relato
df_resultados['sintomas_count'] = df_resultados['sintomas_identificados'].apply(
    lambda x: len(x.split(',')) if x != 'nenhum' else 0
)

# Marca se o diagnóstico foi encontrado (1) ou não (0)
df_resultados['diagnostico_encontrado'] = df_resultados['diagnostico_sugerido'].apply(
    lambda x: 0 if x == 'Não identificado' else 1
)

# Agrega por grupo para comparação
analise = df_resultados.groupby('grupo').agg(
    media_sintomas_identificados=('sintomas_count', 'mean'),
    diagnosticos_encontrados=('diagnostico_encontrado', 'sum'),
    total_pacientes=('paciente', 'count')
).reset_index()

analise['taxa_diagnostico'] = (
    analise['diagnosticos_encontrados'] / analise['total_pacientes'] * 100
).round(1).astype(str) + '%'

print('Análise de viés de gênero na extração de sintomas:\n')
print(analise.to_string(index=False))

print('\nObservação:')
print('Se a taxa de diagnóstico for menor para o perfil feminino atípico,')
print('isso confirma o viés documentado na Fase 1: sintomas atípicos são')
print('menos representados em mapas de conhecimento clínico tradicionais.')

Análise de viés de gênero na extração de sintomas:

           grupo  media_sintomas_identificados  diagnosticos_encontrados  total_pacientes taxa_diagnostico
Feminino atípico                           1.6                         5                5           100.0%
Masculino típico                           1.0                         3                5            60.0%

Observação:
Se a taxa de diagnóstico for menor para o perfil feminino atípico,
isso confirma o viés documentado na Fase 1: sintomas atípicos são
menos representados em mapas de conhecimento clínico tradicionais.


## 7. Exportação dos resultados

In [13]:
# Salva os resultados em CSV para uso nas fases seguintes
df_resultados.to_csv(CAMINHO_SAIDA, index=False, encoding='utf-8')
print(f'Resultados exportados para: {CAMINHO_SAIDA}')

Resultados exportados para: /content/resultados_diagnostico.csv


---

## Considerações finais

Este módulo demonstra como um sistema básico de NLP baseado em correspondência  
de padrões pode apoiar a triagem clínica inicial. As limitações são intencionais  
e fazem parte da análise:

- O sistema depende de correspondência exata de termos — não compreende semântica
- Sintomas atípicos (perfil feminino) são menos representados em mapas de conhecimento tradicionais
- Nas fases seguintes, modelos de ML e NLP mais sofisticados substituirão essa lógica de regras

**Este notebook não produz diagnósticos médicos. Tem finalidade exclusivamente acadêmica.**